# Demo local de CaitSith y CaitSith Studio

Este notebook trabaja contra la version local del repo actual.

Objetivos del demo:
- validar que el notebook usa el paquete `src` de este repositorio;
- ejecutar un flujo representativo sobre `CaitSith`;
- medir cobertura basica del demo frente a la API publica;
- generar CSV de ejemplo para cargarlos despues en CaitSith Studio.

In [9]:
from pathlib import Path
import sys

start_path = Path.cwd().resolve()
candidates = [start_path, *start_path.parents]
package_root = next(
    (path for path in candidates if (path / 'src' / 'caitsith' / 'core.py').exists()),
    None,
)
if package_root is None:
    raise RuntimeError('No se encontro el paquete local de CaitSith desde este notebook.')

src_path = package_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f'package_root = {package_root}')
print(f'src_path = {src_path}')

package_root = C:\Users\alrodriguez\Downloads\caitsith-main\caitsith
src_path = C:\Users\alrodriguez\Downloads\caitsith-main\caitsith\src


In [10]:
import inspect

import pandas as pd
from IPython.display import display

from caitsith import CaitSith
from caitsith_studio.core.operation_registry import OperationRegistry

registry = OperationRegistry.from_caitsith_class(CaitSith)
public_methods = sorted(
    name
    for name, member in inspect.getmembers(CaitSith, predicate=inspect.isfunction)
    if not name.startswith('_')
)

display(
    pd.DataFrame(
        {
            'metrica': [
                'Metodos publicos de CaitSith',
                'Operaciones visibles en CaitSith Studio',
                'Ruta del core cargado',
            ],
            'valor': [
                len(public_methods),
                len(registry.names()),
                inspect.getfile(CaitSith),
            ],
        }
    )
)

,metrica,valor
0,Metodos publicos de CaitSith,262
1,Operaciones visibles en CaitSith Studio,261
2,Ruta del core cargado,C:\Users\alrodriguez\Downloads\caitsith-main\c...


In [11]:
ventas = pd.DataFrame(
    {
        'cliente': ['Ana', 'Luis', 'Ana', 'Marta'],
        'sku': ['A1', 'B2', 'A1', 'C3'],
        'importe': [120.0, 80.0, 150.0, 60.0],
        'coste': [70.0, 55.0, 90.0, 30.0],
        'unidades': [3, 2, 5, 1],
        'precio_unitario': [40.0, 40.0, 30.0, 60.0],
        'pais': ['ES', 'MX', 'ES', 'CO'],
        'fecha_operacion': pd.to_datetime(['2026-04-01', '2026-04-02', '2026-04-05', '2026-04-07']),
    }
)

catalogo = pd.DataFrame(
    {
        'sku': ['A1', 'B2', 'C3'],
        'familia': ['Accesorios', 'Bundles', 'Servicios'],
    }
)

duplicados = pd.DataFrame(
    {
        'cliente': ['Ana', 'Ana', 'Luis'],
        'sku': ['A1', 'A1', 'B2'],
        'importe': [120.0, 120.0, 80.0],
    }
)

display(ventas)
display(catalogo)
display(duplicados)

,cliente,sku,importe,coste,unidades,precio_unitario,pais,fecha_operacion
0,Ana,A1,120.0,70.0,3,40.0,ES,2026-04-01
1,Luis,B2,80.0,55.0,2,40.0,MX,2026-04-02
2,Ana,A1,150.0,90.0,5,30.0,ES,2026-04-05
3,Marta,C3,60.0,30.0,1,60.0,CO,2026-04-07


,sku,familia
0,A1,Accesorios
1,B2,Bundles
2,C3,Servicios


,cliente,sku,importe
0,Ana,A1,120.0
1,Ana,A1,120.0
2,Luis,B2,80.0


In [4]:
cs = CaitSith(ventas.copy())
cs.sumar(['importe', 'coste'], 'importe_mas_coste')
cs.restar(['importe', 'coste'], 'margen')
cs.multiplicar(['unidades', 'precio_unitario'], 'importe_recalculado')
cs.dividir(['importe', 'unidades'], 'ticket_promedio')
cs.si([('margen', '>', 40)], 'alto', 'estandar', 'segmento')
cs.concatenar(['cliente', 'pais'], 'cliente_pais', separator=' | ')
cs.hoy('fecha_hoy')
cs.mes('fecha_operacion', 'mes_operacion')
cs.buscarv('sku', 'familia', 'familia', external_df=catalogo)

resultado = cs.df[
    [
        'cliente',
        'sku',
        'familia',
        'importe',
        'coste',
        'margen',
        'segmento',
        'ticket_promedio',
        'cliente_pais',
        'mes_operacion',
    ]
].copy()

display(resultado)

,cliente,sku,familia,importe,coste,margen,segmento,ticket_promedio,cliente_pais,mes_operacion
0,Ana,A1,Accesorios,120.0,70.0,50.0,alto,40.0,Ana | ES,4
1,Luis,B2,Bundles,80.0,55.0,25.0,estandar,40.0,Luis | MX,4
2,Ana,A1,Accesorios,150.0,90.0,60.0,alto,30.0,Ana | ES,4
3,Marta,C3,Servicios,60.0,30.0,30.0,estandar,60.0,Marta | CO,4


In [5]:
cs_dup = CaitSith(duplicados.copy())
cs_dup.quitar_duplicados(subset=['cliente', 'sku', 'importe'], new_column_name='es_duplicado')
display(cs_dup.df)

,cliente,sku,importe,es_duplicado
0,Ana,A1,120.0,False
1,Ana,A1,120.0,True
2,Luis,B2,80.0,False


In [6]:
demo_methods = {
    'sumar',
    'restar',
    'multiplicar',
    'dividir',
    'si',
    'concatenar',
    'hoy',
    'mes',
    'buscarv',
    'quitar_duplicados',
}

coverage = pd.DataFrame({'metodo': public_methods})
coverage['demo'] = coverage['metodo'].isin(demo_methods)

print(f'Cobertura del notebook: {coverage["demo"].sum()}/{len(coverage)} metodos publicos')
display(coverage[coverage['demo']].reset_index(drop=True))
display(coverage[~coverage['demo']].head(25).reset_index(drop=True))

Cobertura del notebook: 10/262 metodos publicos


,metodo,demo
0,buscarv,True
1,concatenar,True
2,dividir,True
3,hoy,True
4,mes,True
5,multiplicar,True
6,quitar_duplicados,True
7,restar,True
8,si,True
9,sumar,True


,metodo,demo
0,absoluto,False
1,acumulado_por_grupo,False
2,agrupar_transformar,False
3,ahora,False
4,aleatorio,False
5,aleatorio_entre,False
6,ano,False
7,buscar,False
8,buscarh,False
9,buscarh_multiple,False


In [8]:
assets_dir = package_root / 'notebooks' / 'demo_assets'
assets_dir.mkdir(parents=True, exist_ok=True)
ventas.to_csv(assets_dir / 'ventas_demo.csv', index=False)
catalogo.to_csv(assets_dir / 'catalogo_demo.csv', index=False)

print('Archivos listos para cargar en CaitSith Studio:')
print(assets_dir / 'ventas_demo.csv')
print(assets_dir / 'catalogo_demo.csv')

Archivos listos para cargar en CaitSith Studio:
C:\Users\alrodriguez\Downloads\caitsith-main\caitsith\notebooks\demo_assets\ventas_demo.csv
C:\Users\alrodriguez\Downloads\caitsith-main\caitsith\notebooks\demo_assets\catalogo_demo.csv


## Abrir CaitSith Studio

Desde la raiz del proyecto `caitsith/` puedes lanzar la app con:

```bash
streamlit run app.py
```

Puedes cargar los CSV generados en `notebooks/demo_assets/` para repetir en la UI el mismo escenario del notebook.

## Lanzar Studio desde el notebook

Este bloque sirve para:

- arrancar CaitSith Studio desde el propio notebook;
- preparar los DataFrames del notebook como CSV para cargarlos en la app;
- conservar en Python un diccionario `studio_frames` con los DataFrames base;
- volver a ejecutar en el notebook el pipeline exportado desde Studio y recuperar los DataFrames resultantes.

Nota importante: la sesion de Streamlit y la del notebook son procesos distintos. La forma robusta de traer los resultados de la app al notebook es exportar el pipeline a JSON o YAML desde Studio y reejecutarlo aqui.

In [17]:
import socket
import subprocess
import sys
import webbrowser
from pathlib import Path

import pandas as pd

from caitsith import CaitSith
from caitsith_studio.core.executor import PipelineExecutor
from caitsith_studio.core.operation_registry import OperationRegistry
from caitsith_studio.core.serializer import pipeline_from_json, pipeline_from_yaml
from caitsith_studio.models import DataFrameRegistry


def _find_free_port(start_port: int = 8504, max_tries: int = 25) -> int:
    for port in range(start_port, start_port + max_tries):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(("127.0.0.1", port)) != 0:
                return port
    raise RuntimeError("No encontre un puerto libre para Streamlit.")


def _discover_notebook_frames(namespace: dict, frame_names: list[str] | None = None) -> dict[str, pd.DataFrame]:
    if frame_names:
        discovered = {
            name: namespace[name].copy()
            for name in frame_names
            if name in namespace and isinstance(namespace[name], pd.DataFrame)
        }
        if discovered:
            return discovered

    preferred_names = ["ventas", "catalogo", "duplicados", "resultado"]
    discovered = {
        name: namespace[name].copy()
        for name in preferred_names
        if name in namespace and isinstance(namespace[name], pd.DataFrame)
    }
    if discovered:
        return discovered

    excluded = {"coverage"}
    discovered = {
        name: value.copy()
        for name, value in namespace.items()
        if isinstance(value, pd.DataFrame) and not name.startswith("_") and name not in excluded
    }
    if not discovered:
        raise RuntimeError("No encontre DataFrames en el notebook para enviar a Studio.")
    return discovered


def launch_caitsith_studio_from_notebook(
    frame_names: list[str] | None = None,
    *,
    open_browser: bool = True,
    start_port: int = 8504,
    output_dir: Path | None = None,
):
    global studio_process, studio_url, studio_frames, studio_input_dir

    if "studio_process" in globals() and studio_process.poll() is None:
        print(f"Studio ya estaba arrancado en {studio_url}")
        print(f"DataFrames disponibles en memoria: {list(studio_frames)}")
        return studio_process, studio_url, studio_frames

    studio_frames = _discover_notebook_frames(globals(), frame_names=frame_names)
    base_output_dir = output_dir or (assets_dir / "notebook_frames")
    studio_input_dir = Path(base_output_dir)
    studio_input_dir.mkdir(parents=True, exist_ok=True)

    for name, frame in studio_frames.items():
        frame.to_csv(studio_input_dir / f"{name}.csv", index=False)

    studio_port = _find_free_port(start_port=start_port)
    studio_url = f"http://127.0.0.1:{studio_port}"
    studio_command = [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(package_root / "app.py"),
        "--server.headless=true",
        f"--server.port={studio_port}",
    ]
    studio_process = subprocess.Popen(
        studio_command,
        cwd=str(package_root),
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    print(f"Studio lanzado en: {studio_url}")
    print("CSV listos para cargar desde la app:")
    for name in studio_frames:
        print(studio_input_dir / f"{name}.csv")
    print(f"DataFrames disponibles en el notebook: {list(studio_frames)}")

    if open_browser:
        try:
            webbrowser.open(studio_url)
        except Exception:
            pass

    return studio_process, studio_url, studio_frames


def stop_caitsith_studio_from_notebook():
    global studio_process
    if "studio_process" in globals() and studio_process.poll() is None:
        studio_process.terminate()
        print("Studio detenido.")
    else:
        print("No hay ningun proceso de Studio activo.")


studio_process, studio_url, studio_frames = launch_caitsith_studio_from_notebook()

Studio ya estaba arrancado en http://127.0.0.1:8504
DataFrames disponibles en memoria: ['ventas', 'catalogo', 'duplicados', 'resultado']


In [16]:
def run_exported_studio_pipeline(pipeline_path: str | Path, frames: dict[str, pd.DataFrame] | None = None):
    pipeline_path = Path(pipeline_path)
    raw_text = pipeline_path.read_text(encoding="utf-8")

    if pipeline_path.suffix.lower() == ".json":
        steps = pipeline_from_json(raw_text)
    elif pipeline_path.suffix.lower() in {".yml", ".yaml"}:
        steps = pipeline_from_yaml(raw_text)
    else:
        raise ValueError("Usa un archivo .json, .yml o .yaml exportado desde CaitSith Studio.")

    source_frames = (frames or studio_frames).copy()
    dataframe_registry = DataFrameRegistry()
    for name, frame in source_frames.items():
        dataframe_registry.add(name, frame.copy(), source="notebook")

    executor = PipelineExecutor(OperationRegistry.from_caitsith_class(CaitSith))
    result = executor.execute(dataframe_registry, steps)
    return result


def get_studio_frame(frame_name: str, *, result=None) -> pd.DataFrame:
    if result is None:
        return studio_frames[frame_name].copy()
    return result.frames[frame_name].copy()


# Ejemplo de uso despues de exportar el pipeline desde la app:
# studio_result = run_exported_studio_pipeline(package_root / 'notebooks' / 'mi_pipeline.json')
# studio_result.frames.keys()
# get_studio_frame('ventas', result=studio_result).head()

In [18]:
# Ejecuta esta celda para abrir o reutilizar CaitSith Studio desde el notebook.
studio_process, studio_url, studio_frames = launch_caitsith_studio_from_notebook(open_browser=True)

print("Studio URL:", studio_url)
print("Proceso activo:", studio_process.poll() is None)
print("DataFrames en memoria:", list(studio_frames))

# Si el navegador no se abre solo en VS Code, copia esta URL en tu navegador:
studio_url

Studio ya estaba arrancado en http://127.0.0.1:8504
DataFrames disponibles en memoria: ['ventas', 'catalogo', 'duplicados', 'resultado']
Studio URL: http://127.0.0.1:8504
Proceso activo: True
DataFrames en memoria: ['ventas', 'catalogo', 'duplicados', 'resultado']


'http://127.0.0.1:8504'